In [1]:
!pip install pandas numpy seaborn matplotlib scikit-learn glob2 xgboost

  Using cached pandas-3.0.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.6-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached matplotlib-3.10.9-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached scikit_learn-1.9.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached glob2-0.7-py2.py3-none-any.whl
  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached tzdata-2026.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.63.0-cp313-cp313-win_amd64.whl.metadata (121 kB)
  Using cached kiwisolver-1.5.0-cp313-cp313-win_amd64.whl.metadata (5.2 kB)
  Using cached pillow-12.2.0-cp313-cp313-win_amd64.whl.metadata (9.0 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
  Using cached scipy-1.17.


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.model_selection import train_test_split,RandomizedSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix,log_loss

import os
import csv

In [3]:
print("Python is looking inside:", os.getcwd())
print("Does the dataset folder exist there?", os.path.exists(r"..\datasets\American_Sign_Language_Fingerspelling_dataset"))

Python is looking inside: c:\Users\SombatlaTrucDeydeepy\Desktop\Github\SignLink_Honors\sign_lang_classification
Does the dataset folder exist there? True


### Identifying CSV files with more than 51 rows across all user folders

I'll now iterate through each 'User' folder (from User1 to User9) in the `dyfav` directory. For each CSV file found, I'll load it into a Pandas DataFrame and record its `user_id`, `file_index`, and `row_count` if the number of rows exceeds 51.

In [4]:
base_dir = r"..\datasets\American_Sign_Language_Fingerspelling_dataset"

files_with_different_row_counts = []

# Loop through User1 to User9 folders
for i in range(1, 10):
    user_folder_name = f'User{i}'
    user_folder_path = os.path.join(base_dir, user_folder_name)

    if os.path.exists(user_folder_path):
        # Get all csv files in the user's folder
        all_csv_files_in_user_folder = [f for f in os.listdir(user_folder_path) if f.endswith('.csv')]

        for file_index, file_name in enumerate(all_csv_files_in_user_folder):
            file_path = os.path.join(user_folder_path, file_name)
            try:
                df_temp = pd.read_csv(file_path)
                row_count = df_temp.shape[0]

                if row_count > 51:
                    files_with_different_row_counts.append({
                        'user_id': user_folder_name,
                        'file_index': file_index, # 0-based index of the file in the current user's folder
                        'file_name': file_name,
                        'row_count': row_count
                    })
            except Exception as e:
                print(f"Error reading {file_path}: {e}")

print(f"Found {len(files_with_different_row_counts)} files with more than 51 rows.")
if files_with_different_row_counts:
    display(pd.DataFrame(files_with_different_row_counts))
else:
    print("No files found with more than 51 rows.")

Found 10 files with more than 51 rows.


,user_id,file_index,file_name,row_count
0,User1,43,291982821_alphabet_i_right.csv,101
1,User1,95,291982937_alphabet_s_right.csv,101
2,User1,106,291982963_alphabet_u_right.csv,101
3,User1,123,291983028_alphabet_y_right.csv,101
4,User7,7,291996562_alphabet_b_right.csv,101
5,User7,20,291996584_alphabet_e_right.csv,101
6,User7,24,291996590_alphabet_e_right.csv,101
7,User8,50,291981085_alphabet_k_right.csv,101
8,User8,118,291981192_alphabet_x_right.csv,101
9,User8,119,291981193_alphabet_x_right.csv,101


### General Duplicate Check for Files with More Than 51 Rows

Since the specific duplication pattern wasn't found, we'll now perform a general check for *any* duplicate rows within each of the files that have more than 51 rows. This will identify if there are any identical rows appearing more than once in the entire dataset of these files.

In [5]:
general_duplicate_results = []

for index, row in pd.DataFrame(files_with_different_row_counts).iterrows():
    user_folder_name = row['user_id']
    file_name = row['file_name']
    file_path = os.path.join(base_dir, user_folder_name, file_name)

    try:
        df_current = pd.read_csv(file_path)

        # Check for any duplicate rows in the entire DataFrame
        has_any_duplicates = df_current.duplicated().any()
        total_duplicates = df_current.duplicated().sum()

        general_duplicate_results.append({
            'user_id': user_folder_name,
            'file_name': file_name,
            'row_count': df_current.shape[0],
            'has_any_duplicates': has_any_duplicates,
            'total_duplicate_rows': total_duplicates
        })
    except Exception as e:
        print(f"Error processing {file_path}: {e}")

if general_duplicate_results:
    print("General duplicate check complete for files with more than 51 rows:")
    display(pd.DataFrame(general_duplicate_results))
else:
    print("No files with more than 51 rows were found.")

General duplicate check complete for files with more than 51 rows:


,user_id,file_name,row_count,has_any_duplicates,total_duplicate_rows
0,User1,291982821_alphabet_i_right.csv,101,False,0
1,User1,291982937_alphabet_s_right.csv,101,False,0
2,User1,291982963_alphabet_u_right.csv,101,False,0
3,User1,291983028_alphabet_y_right.csv,101,False,0
4,User7,291996562_alphabet_b_right.csv,101,False,0
5,User7,291996584_alphabet_e_right.csv,101,False,0
6,User7,291996590_alphabet_e_right.csv,101,False,0
7,User8,291981085_alphabet_k_right.csv,101,False,0
8,User8,291981192_alphabet_x_right.csv,101,False,0
9,User8,291981193_alphabet_x_right.csv,101,False,0


### Creating a Copy of `dyfav` and Deleting Files

To ensure the integrity of your original data, I'll first create a complete copy of the `dyfav` directory. Then, from this new copied directory, I will delete the CSV files that were identified as having more than 51 rows.

In [7]:
import shutil

original_dyfav_path = r"..\datasets\American_Sign_Language_Fingerspelling_dataset"
copied_dyfav_path = r"..\datasets\American_Sign_Language_Fingerspelling_dataset_copy"

# Create a copy of the dyfav folder
print(f"Copying '{original_dyfav_path}' to '{copied_dyfav_path}'...")
if os.path.exists(copied_dyfav_path):
    shutil.rmtree(copied_dyfav_path)
    print("Existing copy removed.")
shutil.copytree(original_dyfav_path, copied_dyfav_path)
print("Copy complete.")


Copying '..\datasets\American_Sign_Language_Fingerspelling_dataset' to '..\datasets\American_Sign_Language_Fingerspelling_dataset_copy'...
Copy complete.


In [8]:
deleted_files_count = 0

# Iterate through the files identified as having more than 51 rows
for index, row in pd.DataFrame(files_with_different_row_counts).iterrows():
    user_id = row['user_id']
    file_name_to_delete = row['file_name']

    # Construct the path to the file in the *copied* directory
    file_path_in_copy = os.path.join(copied_dyfav_path, user_id, file_name_to_delete)

    if os.path.exists(file_path_in_copy):
        os.remove(file_path_in_copy)
        print(f"Deleted: {file_path_in_copy}")
        deleted_files_count += 1
    else:
        print(f"File not found in copy (already deleted or path error): {file_path_in_copy}")

print(f"\nSuccessfully deleted {deleted_files_count} files from the copied 'dyfav_copy' folder.")


Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User1\291982821_alphabet_i_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User1\291982937_alphabet_s_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User1\291982963_alphabet_u_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User1\291983028_alphabet_y_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User7\291996562_alphabet_b_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User7\291996584_alphabet_e_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User7\291996590_alphabet_e_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User8\291981085_alphabet_k_right.csv
Deleted: ..\datasets\American_Sign_Language_Fingerspelling_dataset_copy\User8\291981192_alphabet_x_right.csv
Deleted: ..\dataset

### Analyzing File Distribution by Alphabet Letter in `dyfav_copy`

Now, let's look at the remaining files in the `dyfav_copy` folder and count how many files correspond to each alphabet letter for every user. This will help us understand the current distribution of data after the deletions.

In [9]:
import re

alphabet_counts = {}
copied_base_dir = '/content/drive/MyDrive/dyfav_copy/'

pd.set_option('display.max_columns', None) # Set display option to show all columns

# Loop through User1 to User9 folders in the copied directory
for i in range(1, 10):
    user_folder_name = f'User{i}'
    user_folder_path = os.path.join(copied_base_dir, user_folder_name)

    if os.path.exists(user_folder_path):
        user_letter_counts = {}

        # Get all csv files in the user's copied folder
        all_csv_files_in_user_folder = [f for f in os.listdir(user_folder_path) if f.endswith('.csv')]

        for file_name in all_csv_files_in_user_folder:
            # Extract the alphabet letter using a regular expression
            match = re.search(r'alphabet_([a-zA-Z])_right\.csv', file_name)
            if match:
                letter = match.group(1).upper() # Get the letter and convert to uppercase
                user_letter_counts[letter] = user_letter_counts.get(letter, 0) + 1

        if user_letter_counts:
            alphabet_counts[user_folder_name] = user_letter_counts

# Prepare data for DataFrame display
data_for_df = []
for user, counts in alphabet_counts.items():
    for letter, count in counts.items():
        data_for_df.append({'User': user, 'Alphabet_Letter': letter, 'File_Count': count})

if data_for_df:
    df_letter_distribution = pd.DataFrame(data_for_df)
    # Pivot the table for better readability
    df_pivot = df_letter_distribution.pivot_table(index='User', columns='Alphabet_Letter', values='File_Count', fill_value=0)
    display(df_pivot)
else:
    print("No CSV files found in the copied 'dyfav_copy' folder to analyze.")

No CSV files found in the copied 'dyfav_copy' folder to analyze.


### Total Files Per User

Let's calculate the total number of files for each user by summing up the counts across all alphabet letters from the `df_pivot` table.

In [10]:
if 'df_pivot' in locals() and not df_pivot.empty:
    total_files_per_user = df_pivot.sum(axis=1).reset_index()
    total_files_per_user.columns = ['User', 'Total_Files']
    print("Total files per user:")
    display(total_files_per_user)
else:
    print("df_pivot DataFrame not found or is empty. Please ensure the previous cell was executed successfully.")

df_pivot DataFrame not found or is empty. Please ensure the previous cell was executed successfully.


### Identifying Unbalanced File Counts for Specific Letters

Based on the table above, it appears that a count of `5.0` is the most common for each letter per user. Let's identify the instances where the file count for a specific letter deviates from this common value.

In [11]:
unbalanced_files = []

# Assuming df_pivot is the DataFrame from the previous step
if 'df_pivot' in locals() and not df_pivot.empty:
    for user in df_pivot.index:
        for letter in df_pivot.columns:
            count = df_pivot.loc[user, letter]
            # Assuming 5.0 is the 'balanced' count
            if count != 5.0:
                unbalanced_files.append({
                    'User': user,
                    'Alphabet_Letter': letter,
                    'File_Count': count
                })

    if unbalanced_files:
        df_unbalanced = pd.DataFrame(unbalanced_files)
        print("Users with unbalanced file counts for specific letters:")
        display(df_unbalanced)
    else:
        print("All file counts appear to be balanced (all 5.0).")
else:
    print("df_pivot DataFrame not found or is empty. Please ensure the previous cell was executed successfully.")

df_pivot DataFrame not found or is empty. Please ensure the previous cell was executed successfully.


SO NO NEED TO DELETE FILES FURTHER. USER IMBALACE IS MAXIMUM +3% AND MINIMUM -2.3% WHICH IS WAY UNDER THE ACCEPTED LIMIT. HOWEVER FOR EACH INDIVUDUAL LETTER THE CLASS IMBALANCE VARIES FROM 40% TO -40% WHICH IS STILL FINE SINCE TREE MODELS HANDLE IT VERY EASILY


### Feature Engineering: Global and Segmented Statistics

This section will perform feature engineering on the CSV files in `dyfav_copy`. For each file, it will calculate:

1.  **Global Features**: `min`, `max`, `mean`, `std_dev`, and `tot_eng` for each of the 17 sensor features over the entire 51 rows.
2.  **Segmented Features**: The same 5 statistics for each of the 17 sensor features, but divided into 5 segments (rows 1-10, 11-20, 21-30, 31-40, 41-51).

Each original CSV file will result in a new CSV file in a new directory (`dybaf_1`) containing a single row with `user_id`, `file_name`, `label`, and all 510 (85 global + 425 segmented) engineered features. A master CSV combining all these features across all files will also be generated.

In [12]:
import re
import shutil

def calculate_features(df_segment, feature_cols, prefix=""):
    """Calculates min, max, mean, std_dev, and total energy for given columns in a DataFrame segment."""
    features = {}
    for col in feature_cols:
        col_prefix = f"{prefix}{col}_" if prefix else f"{col}_"
        features[f"{col_prefix}min"] = df_segment[col].min()
        features[f"{col_prefix}max"] = df_segment[col].max()
        features[f"{col_prefix}mean"] = df_segment[col].mean()

        std_val = df_segment[col].std()
        features[f"{col_prefix}std_dev"] = std_val if not pd.isna(std_val) else 0.0 # Replace NaN std with 0, especially for single-value segments

        features[f"{col_prefix}tot_eng"] = (df_segment[col]**2).sum()
    return features

# Define paths
dyfav_copy_base_dir = '/content/drive/MyDrive/dyfav_copy/'
dybaf_1_base_dir = '/content/drive/MyDrive/dybaf_1/'

# Create the new base directory for engineered features if it doesn't exist
os.makedirs(dybaf_1_base_dir, exist_ok=True)

# Define the 17 sensor feature columns based on notebook context
sensor_features = [
    'emg1', 'emg2', 'emg3', 'emg4', 'emg5', 'emg6', 'emg7', 'emg8',
    'accel_x', 'accel_y', 'accel_z',
    'gyro_x', 'gyro_y', 'gyro_z',
    'roll', 'pitch', 'yaw'
]

all_processed_files_data = [] # To collect all feature sets for a final master DataFrame

print(f"Starting feature engineering from '{dyfav_copy_base_dir}' to '{dybaf_1_base_dir}'...")

# Iterate through User1 to User9 folders in the copied directory
for i in range(1, 10):
    user_folder_name = f'User{i}'
    user_dyfav_copy_path = os.path.join(dyfav_copy_base_dir, user_folder_name)
    user_dybaf_1_path = os.path.join(dybaf_1_base_dir, user_folder_name)

    # Create user subdirectory in dybaf_1
    os.makedirs(user_dybaf_1_path, exist_ok=True)

    if os.path.exists(user_dyfav_copy_path):
        all_csv_files_in_user_folder = [f for f in os.listdir(user_dyfav_copy_path) if f.endswith('.csv')]
        print(f"Processing {len(all_csv_files_in_user_folder)} files in {user_folder_name}...")

        for file_name in all_csv_files_in_user_folder:
            file_path_in_copy = os.path.join(user_dyfav_copy_path, file_name)

            try:
                # Read CSV without a header to ensure integer column names initially
                df = pd.read_csv(file_path_in_copy, header=None)

                num_cols = df.shape[1]
                expected_sensor_cols = len(sensor_features)

                extracted_label = None
                df_sensor = pd.DataFrame() # Initialize df_sensor

                # Process 17-column files (sensor features only) or 18-column files (sensor features + label)
                if num_cols == expected_sensor_cols: # Case: 17 columns (sensor data only)
                    df.columns = sensor_features # Assign sensor feature names
                    df_sensor = df # All columns are sensor features
                elif num_cols == expected_sensor_cols + 1: # Case: 18 columns (sensor data + label)
                    # Assign sensor names and a temporary name for the label column
                    temp_cols = sensor_features + ['temp_label_col']
                    df.columns = temp_cols
                    extracted_label = df['temp_label_col'].iloc[0] # Extract label
                    df_sensor = df[sensor_features] # Select only sensor features for df_sensor
                else:
                    print(f"Warning: Skipping {file_name} in {user_folder_name}. Unexpected number of columns ({num_cols}). Expected 17 or 18.")
                    continue

                # Ensure the DataFrame has 50 or 51 rows (check on df_sensor)
                if df_sensor.shape[0] not in [50, 51]:
                    print(f"Warning: Skipping {file_name} in {user_folder_name}. Expected 50 or 51 rows, but found {df_sensor.shape[0]}.")
                    continue

                # If label is still None (i.e., not extracted from a column), try to extract from filename
                if extracted_label is None:
                    match = re.search(r'alphabet_([a-zA-Z])_right\.csv', file_name)
                    if match:
                        extracted_label = match.group(1).upper()
                    else:
                        print(f"Warning: Skipping {file_name} in {user_folder_name}. Label could not be extracted from column or filename.")
                        continue

                label = extracted_label # Assign to the 'label' variable used later

                # --- Calculate Global Features ---
                global_features = calculate_features(df_sensor, sensor_features, prefix="global_")

                # --- Calculate Segmented Features ---
                segmented_features = {}
                segment_definitions = {
                    "segment1": (0, 10),  # rows 1-10 (0-indexed: 0-9)
                    "segment2": (10, 20), # rows 11-20 (0-indexed: 10-19)
                    "segment3": (20, 30), # rows 21-30 (0-indexed: 20-29)
                    "segment4": (30, 40), # rows 31-40 (0-indexed: 30-39)
                    "segment5": (40, 51)  # rows 41-51 (0-indexed: 40-50). Handles 50 rows by taking df_sensor.iloc[40:50]
                }

                for seg_name, (start_row, end_row) in segment_definitions.items():
                    df_segment = df_sensor.iloc[start_row:end_row]
                    seg_features = calculate_features(df_segment, sensor_features, prefix=f"{seg_name}_")
                    segmented_features.update(seg_features)

                # Combine all features, user_id, file_name, and label
                all_features_for_file = {
                    'user_id': user_folder_name,
                    'file_name': file_name,
                    'label': label,
                    **global_features,
                    **segmented_features
                }

                all_processed_files_data.append(all_features_for_file)

                # Save individual feature file (single row CSV per original file)
                output_file_name = file_name.replace('.csv', '_features.csv') # Append _features to distinguish
                output_file_path = os.path.join(user_dybaf_1_path, output_file_name)

                pd.DataFrame([all_features_for_file]).to_csv(output_file_path, index=False)

            except Exception as e:
                print(f"Error processing {file_name} in {user_folder_name}: {e}")
    else:
        print(f"User folder not found: {user_dyfav_copy_path}")

print("\nFeature engineering complete. Individual feature files saved to 'dybaf_1' folder.")

# Combine all features into a single master DataFrame and save it
if all_processed_files_data:
    master_features_df = pd.DataFrame(all_processed_files_data)
    master_output_path = os.path.join(dybaf_1_base_dir, 'all_users_combined_features.csv')
    master_features_df.to_csv(master_output_path, index=False)
    print(f"All features combined into a master CSV: {master_output_path}")
    print(f"Total features generated per file (excluding metadata): {len(master_features_df.columns) - 3}")
    display(master_features_df.head())
else:
    print("No files were processed for feature engineering.")

Starting feature engineering from '/content/drive/MyDrive/dyfav_copy/' to '/content/drive/MyDrive/dybaf_1/'...
User folder not found: /content/drive/MyDrive/dyfav_copy/User1
User folder not found: /content/drive/MyDrive/dyfav_copy/User2
User folder not found: /content/drive/MyDrive/dyfav_copy/User3
User folder not found: /content/drive/MyDrive/dyfav_copy/User4
User folder not found: /content/drive/MyDrive/dyfav_copy/User5
User folder not found: /content/drive/MyDrive/dyfav_copy/User6
User folder not found: /content/drive/MyDrive/dyfav_copy/User7
User folder not found: /content/drive/MyDrive/dyfav_copy/User8
User folder not found: /content/drive/MyDrive/dyfav_copy/User9

Feature engineering complete. Individual feature files saved to 'dybaf_1' folder.
No files were processed for feature engineering.


In [15]:
df1=pd.read_csv(r"..\datasets\American_Sign_Language_Fingerspelling_dataset\processed_data\pre-pre-processed_sign_lang_values.csv")
df1.head()

,user_id,file_name,label,global_emg1_min,global_emg1_max,global_emg1_mean,global_emg1_std_dev,global_emg1_tot_eng,global_emg2_min,global_emg2_max,global_emg2_mean,global_emg2_std_dev,global_emg2_tot_eng,global_emg3_min,global_emg3_max,global_emg3_mean,global_emg3_std_dev,global_emg3_tot_eng,global_emg4_min,global_emg4_max,global_emg4_mean,global_emg4_std_dev,global_emg4_tot_eng,global_emg5_min,global_emg5_max,global_emg5_mean,global_emg5_std_dev,global_emg5_tot_eng,global_emg6_min,global_emg6_max,global_emg6_mean,global_emg6_std_dev,global_emg6_tot_eng,global_emg7_min,global_emg7_max,global_emg7_mean,global_emg7_std_dev,global_emg7_tot_eng,global_emg8_min,global_emg8_max,global_emg8_mean,global_emg8_std_dev,global_emg8_tot_eng,global_accel_x_min,global_accel_x_max,global_accel_x_mean,global_accel_x_std_dev,global_accel_x_tot_eng,global_accel_y_min,global_accel_y_max,global_accel_y_mean,global_accel_y_std_dev,global_accel_y_tot_eng,global_accel_z_min,global_accel_z_max,global_accel_z_mean,global_accel_z_std_dev,global_accel_z_tot_eng,global_gyro_x_min,global_gyro_x_max,global_gyro_x_mean,global_gyro_x_std_dev,global_gyro_x_tot_eng,global_gyro_y_min,global_gyro_y_max,global_gyro_y_mean,global_gyro_y_std_dev,global_gyro_y_tot_eng,global_gyro_z_min,global_gyro_z_max,global_gyro_z_mean,global_gyro_z_std_dev,global_gyro_z_tot_eng,global_roll_min,global_roll_max,global_roll_mean,global_roll_std_dev,global_roll_tot_eng,global_pitch_min,global_pitch_max,global_pitch_mean,global_pitch_std_dev,global_pitch_tot_eng,global_yaw_min,global_yaw_max,global_yaw_mean,global_yaw_std_dev,global_yaw_tot_eng,segment1_emg1_min,segment1_emg1_max,segment1_emg1_mean,segment1_emg1_std_dev,segment1_emg1_tot_eng,segment1_emg2_min,segment1_emg2_max,segment1_emg2_mean,segment1_emg2_std_dev,segment1_emg2_tot_eng,segment1_emg3_min,segment1_emg3_max,segment1_emg3_mean,segment1_emg3_std_dev,segment1_emg3_tot_eng,segment1_emg4_min,segment1_emg4_max,segment1_emg4_mean,segment1_emg4_std_dev,segment1_emg4_tot_eng,segment1_emg5_min,segment1_emg5_max,segment1_emg5_mean,segment1_emg5_std_dev,segment1_emg5_tot_eng,segment1_emg6_min,segment1_emg6_max,segment1_emg6_mean,segment1_emg6_std_dev,segment1_emg6_tot_eng,segment1_emg7_min,segment1_emg7_max,segment1_emg7_mean,segment1_emg7_std_dev,segment1_emg7_tot_eng,segment1_emg8_min,segment1_emg8_max,segment1_emg8_mean,segment1_emg8_std_dev,segment1_emg8_tot_eng,segment1_accel_x_min,segment1_accel_x_max,segment1_accel_x_mean,segment1_accel_x_std_dev,segment1_accel_x_tot_eng,segment1_accel_y_min,segment1_accel_y_max,segment1_accel_y_mean,segment1_accel_y_std_dev,segment1_accel_y_tot_eng,segment1_accel_z_min,segment1_accel_z_max,segment1_accel_z_mean,segment1_accel_z_std_dev,segment1_accel_z_tot_eng,segment1_gyro_x_min,segment1_gyro_x_max,segment1_gyro_x_mean,segment1_gyro_x_std_dev,segment1_gyro_x_tot_eng,segment1_gyro_y_min,segment1_gyro_y_max,segment1_gyro_y_mean,segment1_gyro_y_std_dev,segment1_gyro_y_tot_eng,segment1_gyro_z_min,segment1_gyro_z_max,segment1_gyro_z_mean,segment1_gyro_z_std_dev,segment1_gyro_z_tot_eng,segment1_roll_min,segment1_roll_max,segment1_roll_mean,segment1_roll_std_dev,segment1_roll_tot_eng,segment1_pitch_min,segment1_pitch_max,segment1_pitch_mean,segment1_pitch_std_dev,segment1_pitch_tot_eng,segment1_yaw_min,segment1_yaw_max,segment1_yaw_mean,segment1_yaw_std_dev,segment1_yaw_tot_eng,segment2_emg1_min,segment2_emg1_max,segment2_emg1_mean,segment2_emg1_std_dev,segment2_emg1_tot_eng,segment2_emg2_min,segment2_emg2_max,segment2_emg2_mean,segment2_emg2_std_dev,segment2_emg2_tot_eng,segment2_emg3_min,segment2_emg3_max,segment2_emg3_mean,segment2_emg3_std_dev,segment2_emg3_tot_eng,segment2_emg4_min,segment2_emg4_max,segment2_emg4_mean,segment2_emg4_std_dev,segment2_emg4_tot_eng,segment2_emg5_min,segment2_emg5_max,segment2_emg5_mean,segment2_emg5_std_dev,segment2_emg5_tot_eng,segment2_emg6_min,segment2_emg6_max,segment2_emg6_mean,segment2_emg6_std_dev,segment2_emg6_tot_eng,segment2_emg7_min,segment2_emg7_max

In [16]:
X = df1.drop(columns=['file_name', 'user_id', 'label'])
y = df1["label"]



In [17]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

### Leave One Group Out (LOGO) Train-Test Split

We will use `LeaveOneGroupOut` to ensure that data from one user is completely held out for testing in each fold. This is a robust way to evaluate model performance on new, unseen users.

In [18]:
from sklearn.model_selection import LeaveOneGroupOut

# Get the 'user_id' column to define the groups
groups = df1['user_id']

# Initialize LeaveOneGroupOut
logo = LeaveOneGroupOut()

print(f"Number of splits (unique users): {logo.get_n_splits(X, y, groups)}\n")

# Iterate through the splits
for train_idx, test_idx in logo.split(X, y, groups):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Get the user ID for the current test set
    test_user_id = groups.iloc[test_idx].unique()[0]

    print(f"Test Group: {test_user_id}")
    print(f"  Training set shape: {X_train.shape}, {y_train.shape}")
    print(f"  Testing set shape: {X_test.shape}, {y_test.shape}")
    print("\n" + "-"*30 + "\n")

print("Each block above represents one iteration of the LOGO split, where data from one user was used for testing, and the rest for training.")

Number of splits (unique users): 9

Test Group: User1
  Training set shape: (1050, 510), (1050,)
  Testing set shape: (128, 510), (128,)

------------------------------

Test Group: User2
  Training set shape: (1046, 510), (1046,)
  Testing set shape: (132, 510), (132,)

------------------------------

Test Group: User3
  Training set shape: (1046, 510), (1046,)
  Testing set shape: (132, 510), (132,)

------------------------------

Test Group: User4
  Training set shape: (1045, 510), (1045,)
  Testing set shape: (133, 510), (133,)

------------------------------

Test Group: User5
  Training set shape: (1044, 510), (1044,)
  Testing set shape: (134, 510), (134,)

------------------------------

Test Group: User6
  Training set shape: (1047, 510), (1047,)
  Testing set shape: (131, 510), (131,)

------------------------------

Test Group: User7
  Training set shape: (1048, 510), (1048,)
  Testing set shape: (130, 510), (130,)

------------------------------

Test Group: User8
  Traini

In [19]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

logo = LeaveOneGroupOut()

accuracies = []

for train_idx, test_idx in logo.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    y_train = y[train_idx]
    y_test = y[test_idx]

    model = XGBClassifier(
        objective='multi:softmax',
        num_class=26,
        max_depth=4,
        n_estimators=200,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)

    accuracies.append(acc)

    print(f"Fold Accuracy: {acc:.4f}")

print(f"\nMean Accuracy: {sum(accuracies)/len(accuracies):.4f}")

Fold Accuracy: 0.2969
Fold Accuracy: 0.2879
Fold Accuracy: 0.2424
Fold Accuracy: 0.0827
Fold Accuracy: 0.1493
Fold Accuracy: 0.2214
Fold Accuracy: 0.1615
Fold Accuracy: 0.1811
Fold Accuracy: 0.2443

Mean Accuracy: 0.2075
